In [1]:
import pandas as pd

In [2]:
envisoft_path = "/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"
iqair_path = "/home/slow_data/Air_Quality/IQAir_air_quality.csv"

envisoft_output_path = "/home/work1/projects/Air_Quality/Masterdata/envisoft_stations.csv"
iqair_output_path = "/home/work1/projects/Air_Quality/Masterdata/iqair_stations.csv"

In [3]:
envisoft_df = pd.read_csv(envisoft_path)
iqair_df = pd.read_csv(iqair_path)

In [4]:
envisoft_df

,Source,ID,Timestamp,Name,Latitude,Longitude,AQI,PM2.5,PM10,CO,NO2,O3,SO2,Temperature,Humidity,Pressure,Wind Speed
0,gov,28560877461938780203765592307,08/04/2025 14:00,Hà Nội: 556 Nguyễn Văn Cừ (KK),21.0491,105.88310,134,134.333871,80.967371,8.601812,52.59890,8.319781,5.55468,27.96,71.0,1012.0,3.90
1,gov,31390912357075263208060500522,08/04/2025 14:00,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,10.7823,106.75280,95,95.340935,59.552588,NaN,8.45665,NaN,16.50532,35.01,46.0,1009.0,1.54
2,gov,31390932574706768021562473002,08/04/2025 14:00,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,10.5391,106.40450,89,88.681186,52.501889,29.512184,NaN,NaN,1.95932,29.45,53.0,1009.0,5.84
3,gov,31390903576425084107499649578,08/04/2025 14:00,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),21.0052,105.84180,151,151.418019,72.311020,NaN,12.98290,9.472219,2.94200,28.00,70.0,1012.0,3.85
4,gov,31390908889087377344742439468,08/04/2025 14:00,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,21.0031,105.79470,107,106.968661,71.106262,12.374913,4.44125,9.902344,2.29400,28.01,69.0,1012.0,3.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84857,gov,31387251434693138681789561386,18/03/2026 08:00,Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...,21.3015,106.22603,47,46.630328,42.558083,NaN,NaN,NaN,9.77068,24.58,74.0,1016.0,3.34
84858,gov,31388883344354363840031242796,18/03/2026 08:00,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,20.5360,105.91650,56,38.423141,NaN,9.886183,32.12790,15.395469,4.17668,25.17,68.0,1015.0,3.65
84859,gov,31390932574706768021562473002,18/03/2026 08:00,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,10.5391,106.40450,31,31.275608,NaN,NaN,NaN,19.987750,1.87768,28.41,58.0,1014.0,4.87
84860,gov,31388839920718814259329251882,18/03/2026 08:00,"Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...",10.9923,106.65770,39,21.326440,39.344582,14.803187,11.23790,NaN,0.67824,28.03,57.0,1014.0,5.66


In [5]:
envisoft_station_df = (
    envisoft_df[['ID', 'Name', 'Latitude', 'Longitude']]
    .drop_duplicates(subset='ID')
)
envisoft_station_df = envisoft_station_df.reset_index(drop=True).sort_values('Name')

In [6]:
import re

# Mapping tên tỉnh/thành → region
REGION_MAP = {
    # Bắc
    'Bắc Giang': 'Bắc',
    'Hà Nam': 'Bắc',
    'Hà Nội': 'Bắc',
    'Phú Thọ': 'Bắc',
    'Thái Nguyên': 'Bắc',
    # Trung
    'Quảng Bình': 'Trung',
    'Đà Nẵng': 'Trung',
    # Nam
    'Bình Dương': 'Nam',
    'HCM': 'Nam',
    'Long An': 'Nam',
}

def normalize_name(name: str) -> str:
    """
    Chuẩn hóa tên trạm:
    - Loại bỏ '(KK)' ở cuối
    - Chuẩn hóa viết hoa đầu từ
    - Trim khoảng trắng thừa
    """
    # Bỏ hậu tố (KK)
    name = re.sub(r'\s*\(KK\)\s*$', '', name).strip()

    # Chuẩn hóa TP./P./Đ. (giữ nguyên viết tắt, chỉ strip khoảng trắng)
    name = re.sub(r'\s+', ' ', name)

    return name

def extract_province(name: str) -> str:
    """Trích tên tỉnh/thành từ đầu tên trạm (trước dấu ':')."""
    if ':' in name:
        province = name.split(':')[0].strip()
        # Chuẩn hóa 'Thái nguyên' → 'Thái Nguyên'
        province = province.title()
        # Giữ lại 'HCM' không title-case
        if province.upper() in ('HCM', 'TP.HCM'):
            province = 'HCM'
        return province
    return ''

def get_region(name: str) -> str:
    province = extract_province(name)
    return REGION_MAP.get(province, 'Không xác định')

# Áp dụng
envisoft_station_df['Name'] = envisoft_station_df['Name'].apply(normalize_name)
envisoft_station_df['Region'] = envisoft_station_df['Name'].apply(get_region)
envisoft_station_df = envisoft_station_df.sort_values(by=["Region", "Name"])

In [7]:
envisoft_station_df.rename(columns={'Name': 'station_name', "Latitude": "latitude", "Longitude": "Longitude", "Region": "region"}, inplace=True)

In [8]:
envisoft_station_df

,ID,station_name,latitude,Longitude,region
8,31387251434693138681789561386,Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...,21.30150,106.22603,Bắc
9,31388883344354363840031242796,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,20.53600,105.91650,Bắc
0,28560877461938780203765592307,Hà Nội: 556 Nguyễn Văn Cừ,21.04910,105.88310,Bắc
4,31390908889087377344742439468,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến,21.00310,105.79470,Bắc
3,31390903576425084107499649578,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng,21.00520,105.84180,Bắc
7,28505268571336961948594948504,Phú Thọ: đường Hùng Vương - Tp Việt Trì,21.33847,105.36330,Bắc
6,29195707587706641566224751462,Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên,21.59315,105.84310,Bắc
10,31388839920718814259329251882,"Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...",10.99230,106.65770,Nam
12,31390916083317566102523755051,HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trư...,10.78230,106.68340,Nam
1,31390912357075263208060500522,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,10.78230,106.75280,Nam


In [9]:
iqair_df

,timestamp,station_name,longitude,latitude,aqi,WHO_exposure,PM2.5 (µg/m³),PM10 (µg/m³),O3 (µg/m³),NO2 (µg/m³),SO2 (µg/m³),CO (µg/m³),condition,temperature (°),humidity (%),pressure,wind_speed (km/h),wind_direction
0,2025-04-17 01:00:00,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.841800,21.005200,167,15.6,78.1,215.0,62.8,9.6,8.2,NaN,Nhiều mây,23,86,1007,13.2,135
1,2025-04-17 01:00:00,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.794700,21.003100,154,12.1,60.3,204.5,10.2,2.5,6.5,2.0,Nhiều mây,23,86,1007,13.4,136
2,2025-04-17 01:00:00,Minh Khai - Bắc Từ Liêm,105.740000,21.050000,132,5.2,26.2,218.7,21.0,NaN,0.1,1.4,Mưa,23,84,1007,12.3,132
3,2025-04-17 01:00:00,Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK),107.084400,10.367976,83,5.2,26.2,66.6,73.5,3.9,6.2,0.1,Nhiều mây,27,83,1010,19.0,103
4,2025-04-17 01:00:00,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.335700,20.938100,144,10.6,53.0,147.4,48.2,1.0,1.3,2.2,Nhiều mây,21,88,1007,11.7,119
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84734,2026-01-08 09:00:00,Thừa Thiên Huế: 83 đường Hùng Vương (KK),107.596351,16.462260,66,3.5,17.5,23.6,54.3,NaN,121.0,2.2,Sương mù,16,94,1024,13.0,280
84735,2026-01-08 09:00:00,Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông -...,106.343900,20.457800,189,22.0,109.8,154.1,20.8,75.8,10.4,2.5,Trời quang,14,67,1028,9.9,334
84736,2026-01-08 09:00:00,Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên...,105.843104,21.593151,172,17.0,85.0,24.7,NaN,NaN,5.9,0.8,Nhiều mây,13,63,1028,5.3,355
84737,2026-01-08 09:00:00,Quảng Bình: Khu kinh tế Hòn La (KK),106.496620,17.932938,54,2.2,10.8,15.8,10.0,NaN,13.4,2.6,Nhiều mây,18,58,1026,30.3,11


In [10]:
cols = ['station_name', 'longitude', 'latitude']

iqair_station_df = (
    iqair_df[cols]
    .drop_duplicates(subset='station_name')
)

# Round tọa độ để gom các điểm gần nhau (sai số ~100m)
iqair_station_df = iqair_station_df.copy()
iqair_station_df['_lat_r'] = iqair_station_df['latitude'].round(3)
iqair_station_df['_lon_r'] = iqair_station_df['longitude'].round(3)

iqair_station_df = iqair_station_df.sort_values(
    'station_name', key=lambda s: s.str.len(), ascending=False
)
iqair_station_df = (
    iqair_station_df
    .drop_duplicates(subset=['_lat_r', '_lon_r'])
    .drop(columns=['_lat_r', '_lon_r'])
    .reset_index(drop=True)
)

In [11]:
iqair_station_df = iqair_station_df[
    iqair_station_df['station_name'] != 'Office IQAir Ha Noi'
].reset_index(drop=True)

In [12]:
iqair_station_df

,station_name,longitude,latitude
0,Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông -...,106.343900,20.457800
1,Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm...,105.852771,21.035584
2,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.841800,21.005200
3,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.794700,21.003100
4,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.335700,20.938100
5,Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên...,105.843104,21.593151
6,Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK),107.084400,10.367976
7,Thừa Thiên Huế: 83 đường Hùng Vương (KK),107.596351,16.462260
8,Quảng Bình: Khu kinh tế Hòn La (KK),106.496620,17.932938
9,IQAir Vietnam - Saigon Pearl,106.718700,10.790500


In [13]:
REGION_MAP = {
    # Bắc
    'Hà Nội':        'Bắc',
    'Thái Nguyên':   'Bắc',
    'Thái Bình':     'Bắc',
    'Hải Dương':     'Bắc',
    # Trung
    'Quảng Bình':    'Trung',
    'Thừa Thiên Huế':'Trung',
    # Nam
    'Vũng Tàu':      'Nam',
    'Trà Vinh':      'Nam',
}

# Map thủ công cho các tên không có prefix "Tỉnh:"
MANUAL_MAP = {
    'Minh Khai - Bắc Từ Liêm':       'Bắc',
    'IQAir Ha Noi':                   'Bắc',
    'HCM - FPT Thuduc':               'Nam',
    'IQAir Vietnam - Saigon Pearl':   'Nam',
}

def get_region(name: str) -> str:
    if name in MANUAL_MAP:
        return MANUAL_MAP[name]
    if ':' in name:
        province = name.split(':')[0].strip().title()
        return REGION_MAP.get(province, 'Không xác định')
    return 'Không xác định'

iqair_station_df['region'] = iqair_station_df['station_name'].apply(get_region)
iqair_station_df = iqair_station_df.sort_values(by=["region", "station_name"])

In [14]:
iqair_station_df

,station_name,longitude,latitude,region
11,Hà Nội: Chi cục BVMT (KK),105.800130,21.015250,Bắc
3,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.794700,21.003100,Bắc
1,Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm...,105.852771,21.035584,Bắc
2,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.841800,21.005200,Bắc
4,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.335700,20.938100,Bắc
14,IQAir Ha Noi,105.826200,21.067900,Bắc
12,Minh Khai - Bắc Từ Liêm,105.740000,21.050000,Bắc
0,Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông -...,106.343900,20.457800,Bắc
5,Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên...,105.843104,21.593151,Bắc
13,HCM - FPT Thuduc,106.809100,10.841600,Nam


In [15]:
iqair_station_df.to_csv(iqair_output_path, index=False)
envisoft_station_df.to_csv(envisoft_output_path, index=False)